# Student Performance Analysis and Prediction System Using Data Mining
**Course:** Data Mining (Academic Project)  
**Techniques:** Data Preprocessing, Exploratory Data Analysis, K-Means Clustering (Unsupervised), Decision Tree Classification (Supervised)  
**Dataset:** UCI Machine Learning Repository - Student Performance (Mathematics: `student-mat.csv`)


## 1. Project Overview & Objectives
The primary objective of this project is to apply fundamental data mining techniques to analyze student academic performance records, uncover behavioral patterns, segment students using clustering, and train a supervised decision tree model to predict whether a student will pass or fail their final examination.

### Academic Syllabus Concepts Demonstrated:
1. **Data Cleaning & Transformation:** Imputation of missing values and categorical encoding.
2. **Target Leakage Prevention:** Strictly excluding the final grade `G3` from input predictors.
3. **Unsupervised Learning (K-Means Clustering):** Segmenting students without target guidance.
4. **Supervised Learning (Decision Tree Classifier):** Building a rule-based predictive classifier using Information Gain (Entropy).
5. **Model Evaluation:** Computing Accuracy, Precision, Recall, F1-Score, and Confusion Matrix.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization theme
sns.set_theme(style='whitegrid', palette='Blues_r')
plt.rcParams['figure.dpi'] = 120

print('Environment setup completed.')


## 2. Dataset Loading & Inspection
We load `student-mat.csv` from the `data/` directory. The dataset uses semicolons (`;`) as delimiters.


In [ ]:
# Load the dataset
data_path = os.path.join('..', 'data', 'student-mat.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'student-mat.csv')

df = pd.read_csv(data_path, sep=';')
print(f'Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


In [ ]:
# Inspect attribute types and missing values
df.info()
print('\nMissing values per column:\n', df.isnull().sum())
print(f'Total duplicates found: {df.duplicated().sum()}')


## 3. Data Preprocessing & Target Formulation
### Target Variable Creation:
In the Portuguese educational system, final grade `G3` ranges from 0 to 20. A score of 10 or above is considered passing:
- **Pass (1):** $G3 \ge 10$
- **Fail (0):** $G3 < 10$

### Target Leakage Prevention:
`G3` is strictly removed from input features. Earlier exam results (`G1` and `G2`) are retained as historical exam predictors.


In [ ]:
# Create binary target
df['passed'] = (df['G3'] >= 10).astype(int)
print('Target Class Distribution:')
print(df['passed'].value_counts())
print('\nPercentage Distribution:')
print(df['passed'].value_counts(normalize=True).round(3) * 100)


## 4. Exploratory Data Analysis (EDA)
We visualize the key relationships in the dataset.


In [ ]:
# 1. Distribution of Final Grades (G3)
plt.figure(figsize=(8, 4.5))
sns.histplot(df['G3'], bins=20, kde=True, color='#1f77b4', edgecolor='black')
plt.axvline(10, color='red', linestyle='--', linewidth=2, label='Passing Threshold (10)')
plt.title('Distribution of Final Grades (G3)', fontsize=13, fontweight='bold')
plt.xlabel('Final Grade G3 (0 - 20)')
plt.ylabel('Student Count')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 2. Pass vs Fail Distribution
plt.figure(figsize=(6, 4))
counts = df['passed'].map({1: 'Passed (≥10)', 0: 'Failed (<10)'}).value_counts()
sns.barplot(x=counts.index, y=counts.values, palette=['#1f77b4', '#d62728'], edgecolor='black')
plt.title('Proportion of Passing vs Failing Students', fontsize=13, fontweight='bold')
plt.ylabel('Number of Students')
plt.tight_layout()
plt.show()


In [ ]:
# 3. Weekly Study Time vs Final Grade
study_labels = {1: '< 2 hrs', 2: '2 - 5 hrs', 3: '5 - 10 hrs', 4: '> 10 hrs'}
df_plot = df.copy()
df_plot['study_label'] = df_plot['studytime'].map(study_labels)

plt.figure(figsize=(8, 4.5))
sns.boxplot(x='study_label', y='G3', data=df_plot, order=['< 2 hrs', '2 - 5 hrs', '5 - 10 hrs', '> 10 hrs'], palette='Blues', hue='study_label', legend=False)
plt.axhline(10, color='red', linestyle='--', label='Pass Line (10)')
plt.title('Weekly Study Time vs Final Grade (G3)', fontsize=13, fontweight='bold')
plt.xlabel('Study Time Tier')
plt.ylabel('Final Grade (0 - 20)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 4. Correlation Heatmap of Numerical Features
num_cols = ['age', 'studytime', 'failures', 'absences', 'G1', 'G2', 'G3']
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Key Numerical Attributes', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 5. K-Means Clustering (Unsupervised Learning)
We group students into natural cohorts using K-Means based on:
`studytime`, `failures`, `absences`, `G1`, and `G2`.
We use `StandardScaler` to normalize the scales.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_features = ['studytime', 'failures', 'absences', 'G1', 'G2']
X_cluster = df[cluster_features]

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Elbow Method (WCSS / Inertia)
inertias = []
k_range = range(1, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_range, inertias, marker='o', linewidth=2, color='#1f77b4')
plt.scatter([3], [inertias[2]], color='red', s=120, zorder=5, label='Optimal Elbow (k=3)')
plt.title('Elbow Method for Optimal Number of Clusters', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares (Inertia)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Fit K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Objective Cluster Analysis based on actual mean grades
cluster_profiles = df.groupby('cluster')[['G1', 'G2', 'G3', 'failures', 'studytime', 'absences']].mean()
cluster_profiles['Student Count'] = df['cluster'].value_counts()
print('Cluster Academic Profiles:')
display(cluster_profiles.round(2))


In [ ]:
# Scatter Plot: G1 vs G2 colored by Cluster
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='G1', y='G2', hue='cluster', palette='Set1', s=60, alpha=0.8)
plt.title('K-Means Student Clusters (Period 1 vs Period 2 Exam Performance)', fontsize=13, fontweight='bold')
plt.xlabel('Period 1 Grade (G1)')
plt.ylabel('Period 2 Grade (G2)')
plt.legend(title='Cluster ID')
plt.tight_layout()
plt.show()


## 6. Supervised Decision Tree Classification
We train a `DecisionTreeClassifier` using **Information Gain (Entropy)** to predict `passed` (1 or 0).
We build a full Scikit-learn `Pipeline` to prevent any data leakage.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Define features
num_features = ['age', 'studytime', 'failures', 'absences', 'G1', 'G2']
cat_features = ['sex', 'address', 'schoolsup', 'famsup', 'paid', 'activities', 'higher', 'internet']
feature_cols = num_features + cat_features

X = df[feature_cols]
y = df['passed']

# 80:20 Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Build Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), num_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ]
)

# Combine Preprocessor with Decision Tree Classifier
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(criterion='entropy', max_depth=4, min_samples_leaf=5, random_state=42))
])

# Fit on training data only
pipeline.fit(X_train, y_train)
print('Decision Tree model successfully trained on training set.')


## 7. Model Evaluation
We test the model on the unseen test set (79 students) and evaluate its performance.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_pred = pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
print(f'Precision: {prec:.4f} ({prec*100:.2f}%)')
print(f'Recall:    {rec:.4f} ({rec*100:.2f}%)')
print(f'F1-Score:  {f1:.4f} ({f1*100:.2f}%)\n')

print('Classification Report:\n', classification_report(y_test, y_pred, target_names=['Fail (0)', 'Pass (1)']))


In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Fail (0)', 'Predicted Pass (1)'],
            yticklabels=['Actual Fail (0)', 'Actual Pass (1)'])
plt.title('Confusion Matrix: Student Pass/Fail Prediction', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()


In [ ]:
# Visualize the Decision Tree
plt.figure(figsize=(18, 9))
plot_tree(
    pipeline.named_steps['classifier'],
    feature_names=[f.replace('num__', '').replace('cat__', '') for f in pipeline.named_steps['preprocessor'].get_feature_names_out()],
    class_names=['Fail', 'Pass'],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title('Trained Decision Tree (Information Gain / Entropy Splits)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Sample Predictions on New Students
We test inference on two distinct student profiles:
1. High-performing, diligent student.
2. At-risk student with prior failures.


In [ ]:
student_a = pd.DataFrame([{
    'age': 16, 'sex': 'F', 'address': 'U', 'studytime': 3, 'failures': 0,
    'schoolsup': 'no', 'famsup': 'yes', 'paid': 'no', 'activities': 'yes',
    'higher': 'yes', 'internet': 'yes', 'absences': 2, 'G1': 14, 'G2': 15
}])

student_b = pd.DataFrame([{
    'age': 18, 'sex': 'M', 'address': 'R', 'studytime': 1, 'failures': 2,
    'schoolsup': 'yes', 'famsup': 'no', 'paid': 'no', 'activities': 'no',
    'higher': 'no', 'internet': 'no', 'absences': 18, 'G1': 6, 'G2': 7
}])

pred_a = pipeline.predict(student_a)[0]
prob_a = pipeline.predict_proba(student_a)[0]

pred_b = pipeline.predict(student_b)[0]
prob_b = pipeline.predict_proba(student_b)[0]

label_a = 'PASS' if pred_a == 1 else 'FAIL'
label_b = 'PASS' if pred_b == 1 else 'FAIL'

print(f'Student A Prediction: {label_a} (Confidence: {max(prob_a):.1%})')
print(f'Student B Prediction: {label_b} (Confidence: {max(prob_b):.1%})')


## 9. Conclusion & Academic Summary
In this project, we successfully applied key data mining techniques:
1. **Data Cleaning & Preprocessing:** Cleaned the UCI Mathematics student records, converted categorical attributes, and isolated final grade `G3` to prevent target leakage.
2. **Exploratory Data Analysis:** Uncovered that previous examination results (`G1` and `G2`) are the most influential predictors, and that study time positively correlates with academic success.
3. **K-Means Clustering:** Discovered 3 distinct student segments (High Achievers, Average Performers, and At-Risk Students).
4. **Decision Tree Classification:** Built an interpretable Information Gain (Entropy) decision tree achieving ~87% accuracy with 100% precision on passing students.
